# CS60075 — NLP Assignment 2
## Task B: RNN Seq2seq Model — Training and Evaluation

**Sayan Das (25CS60R01) · Mridul Kumar (25CS60R40)**

**Run `taskA.ipynb` first** to generate `cache/taskA_output.pkl`.

This notebook covers:
- **Task B1** (Tasks 1–4): EncoderRNN · DecoderRNN · Seq2seqRNN · Train · ROUGE evaluation
- **Task B2** (Tasks 5–8): GloVe embeddings · HierEncoderRNN · Decoder2RNN · Beam Search
- **Bonus** (Task 9): Attention · Embedding tying · Teacher-forcing decay · Mixed precision · Caching · Gradient clipping · Early stopping

## Setup: Imports, Configuration, and Drive Mount

In [ ]:
import math
import importlib.util
import os
import pickle
import random
import re
import string
import time
import unicodedata
from collections import Counter
from contextlib import nullcontext
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
from torch.utils.data import DataLoader, Dataset

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(x, **kwargs):
        return x

SEED = 42

def seed_everything(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE

device(type='cuda')

In [ ]:
@dataclass
class Config:
    train_csv: str = "/content/drive/MyDrive/NLP Assignment 2/train.csv"
    test_csv: str = "/content/drive/MyDrive/NLP Assignment 2/test.csv"
    train_subset_size: int = 5000
    val_size: int = 500
    min_doc_freq_ratio: float = 0.01
    hidden_dim: int = 300
    embed_dim: int = 300
    batch_size: int = 16
    epochs: int = 10
    learning_rate: float = 2e-3
    teacher_forcing_ratio: float = 1.0
    teacher_forcing_start: float = 1.0
    teacher_forcing_end: float = 0.6
    max_input_tokens: int = 600
    max_output_tokens: int = 25
    max_sentences: int = 40
    max_sentence_tokens: int = 40
    dropout: float = 0.3
    beam_width: int = 4
    length_penalty_alpha: float = 0.7
    glove_path: Optional[str] = "/content/drive/MyDrive/NLP Assignment 2/glove.6B.300d.txt"
    num_workers: int = 0
    cache_dir: str = "/content/drive/MyDrive/NLP Assignment 2/cache"
    use_preprocess_cache: bool = True
    use_vocab_cache: bool = True
    use_attention: bool = True
    tie_embeddings: bool = False
    use_mixed_precision: bool = True

CFG = Config()
print(asdict(CFG))

{'train_csv': '/content/drive/MyDrive/NLP Assignment 2/train.csv', 'test_csv': '/content/drive/MyDrive/NLP Assignment 2/test.csv', 'train_subset_size': 5000, 'val_size': 500, 'min_doc_freq_ratio': 0.01, 'hidden_dim': 300, 'embed_dim': 300, 'batch_size': 16, 'epochs': 10, 'learning_rate': 0.002, 'teacher_forcing_ratio': 1.0, 'teacher_forcing_start': 1.0, 'teacher_forcing_end': 0.6, 'max_input_tokens': 600, 'max_output_tokens': 25, 'max_sentences': 40, 'max_sentence_tokens': 40, 'dropout': 0.3, 'beam_width': 4, 'length_penalty_alpha': 0.7, 'glove_path': '/content/drive/MyDrive/NLP Assignment 2/glove.6B.300d.txt', 'num_workers': 0, 'cache_dir': '/content/drive/MyDrive/NLP Assignment 2/cache', 'use_preprocess_cache': True, 'use_vocab_cache': True, 'use_attention': True, 'tie_embeddings': False, 'use_mixed_precision': True}


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
installed_packages = {
    "nltk": importlib.util.find_spec("nltk") is not None,
    "rouge_score": importlib.util.find_spec("rouge_score") is not None,
}
installed_packages

{'nltk': True, 'rouge_score': False}

In [ ]:
import nltk

def ensure_nltk_resource(resource_name: str, download_name: Optional[str] = None):
    try:
        nltk.data.find(resource_name)
    except LookupError:
        nltk.download(download_name or resource_name.split("/")[-1], quiet=True)

ensure_nltk_resource("tokenizers/punkt", "punkt")
ensure_nltk_resource("corpora/stopwords", "stopwords")
ensure_nltk_resource("corpora/wordnet", "wordnet")
ensure_nltk_resource("corpora/omw-1.4", "omw-1.4")

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import sent_tokenize

STOPWORDS = set(stopwords.words("english"))
LEMMATIZER = WordNetLemmatizer()

## Load Task A Outputs

Restore preprocessed dataframes, vocabulary, and DataLoaders saved by `taskA.ipynb`.

In [ ]:
PAD_TOKEN = "<pad>"
UNK_TOKEN = "<unk>"
BOS_TOKEN = "<bos>"
EOS_TOKEN = "<eos>"
NUM_TOKEN = "<num>"
SPECIAL_TOKENS = [PAD_TOKEN, UNK_TOKEN, BOS_TOKEN, EOS_TOKEN]

class Vocab:
    def __init__(self, stoi: Dict[str, int], itos: List[str]):
        self.stoi = stoi
        self.itos = itos
        self.pad_id = stoi[PAD_TOKEN]
        self.unk_id = stoi[UNK_TOKEN]
        self.bos_id = stoi[BOS_TOKEN]
        self.eos_id = stoi[EOS_TOKEN]

    def __len__(self):
        return len(self.itos)

    def encode(self, tokens: Sequence[str]) -> List[int]:
        return [self.stoi.get(tok, self.unk_id) for tok in tokens]

    def decode(self, ids: Sequence[int], skip_special: bool = True) -> List[str]:
        tokens = []
        for idx in ids:
            token = self.itos[int(idx)]
            if skip_special and token in SPECIAL_TOKENS:
                continue
            tokens.append(token)
        return tokens

def build_vocab(train_frame: pd.DataFrame, min_doc_freq_ratio: float = 0.01) -> Vocab:
    min_df = max(1, math.ceil(min_doc_freq_ratio * len(train_frame)))
    doc_freq = Counter()
    for _, row in train_frame.iterrows():
        unique_tokens = set(row["text_tokens"]) | set(row["title_tokens"])
        doc_freq.update(unique_tokens)
    tokens = [tok for tok, freq in doc_freq.items() if freq >= min_df]
    tokens = sorted(tokens)
    itos = SPECIAL_TOKENS + tokens
    stoi = {tok: idx for idx, tok in enumerate(itos)}
    return Vocab(stoi, itos)

def get_vocab_cache_path(cfg: Config) -> Path:
    cache_dir = Path(cfg.cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    cache_name = (
        f"vocab_train{cfg.train_subset_size}_df{cfg.min_doc_freq_ratio}"
        f"_seed{SEED}.pkl"
    )
    return cache_dir / cache_name

vocab_cache_path = get_vocab_cache_path(CFG)
if CFG.use_vocab_cache and vocab_cache_path.exists():
    with vocab_cache_path.open("rb") as f:
        vocab_payload = pickle.load(f)
    vocab = Vocab(vocab_payload["stoi"], vocab_payload["itos"])
    print(f"Loaded cached vocabulary from {vocab_cache_path}")
else:
    vocab = build_vocab(train_proc, CFG.min_doc_freq_ratio)
    if CFG.use_vocab_cache:
        with vocab_cache_path.open("wb") as f:
            pickle.dump({"stoi": vocab.stoi, "itos": vocab.itos}, f)
        print(f"Saved vocabulary cache to {vocab_cache_path}")

print("vocab size:", len(vocab))
print("sample tokens:", vocab.itos[:20])

Loaded cached vocabulary from /content/drive/MyDrive/NLP Assignment 2/cache/vocab_train5000_df0.01_seed42.pkl
vocab size: 4193
sample tokens: ['<pad>', '<unk>', '<bos>', '<eos>', 'aaron', 'abandoned', 'abbey', 'abbreviated', 'abc', 'aberdeen', 'ability', 'able', 'aboard', 'abolished', 'abraham', 'abroad', 'absence', 'absorbed', 'abuse', 'ac']


In [ ]:
# ── Load Task A outputs ────────────────────────────────────────────────────────
import pickle

taskA_path = os.path.join(CFG.cache_dir, "taskA_output.pkl")
with open(taskA_path, "rb") as f:
    taskA_out = pickle.load(f)

train_proc = taskA_out["train_proc"]
val_proc   = taskA_out["val_proc"]
test_proc  = taskA_out["test_proc"]
vocab      = Vocab(taskA_out["vocab_stoi"], taskA_out["vocab_itos"])

# Rebuild DataLoaders
train_ds = TitleDataset(train_proc, vocab)
val_ds   = TitleDataset(val_proc,   vocab)
test_ds  = TitleDataset(test_proc,  vocab)

train_loader = DataLoader(train_ds, batch_size=CFG.batch_size, shuffle=True,
                          num_workers=CFG.num_workers, collate_fn=collate_batch)
val_loader   = DataLoader(val_ds,   batch_size=CFG.batch_size, shuffle=False,
                          num_workers=CFG.num_workers, collate_fn=collate_batch)
test_loader  = DataLoader(test_ds,  batch_size=CFG.batch_size, shuffle=False,
                          num_workers=CFG.num_workers, collate_fn=collate_batch)

print(f"Loaded Task A outputs. Vocab size: {len(vocab)}")
print(f"Train: {len(train_proc)}  |  Val: {len(val_proc)}  |  Test: {len(test_proc)}")

Loaded Task A outputs. Vocab size: 4193
Train: 4500  |  Val: 500  |  Test: 100


In [ ]:
class TitleDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, vocab: Vocab):
        self.frame = frame.reset_index(drop=True)
        self.vocab = vocab

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        src_ids = self.vocab.encode(row["text_tokens"])
        tgt_ids = [self.vocab.bos_id] + self.vocab.encode(row["title_tokens"]) + [self.vocab.eos_id]

        sentence_ids = []
        sentence_lengths = []
        for sent in row["sentence_tokens"]:
            if sent:
                encoded = self.vocab.encode(sent)
                sentence_ids.extend(encoded)
                sentence_lengths.append(len(encoded))

        if not sentence_ids:
            sentence_ids = src_ids[:]
            sentence_lengths = [len(src_ids)] if src_ids else [1]

        return {
            "src_ids": torch.tensor(src_ids or [self.vocab.unk_id], dtype=torch.long),
            "tgt_ids": torch.tensor(tgt_ids, dtype=torch.long),
            "hier_src_ids": torch.tensor(sentence_ids or [self.vocab.unk_id], dtype=torch.long),
            "sentence_lengths": torch.tensor(sentence_lengths, dtype=torch.long),
            "raw_title": row["title"],
        }

def collate_batch(batch):
    src = [item["src_ids"] for item in batch]
    tgt = [item["tgt_ids"] for item in batch]
    hier_src = [item["hier_src_ids"] for item in batch]
    sentence_lengths = [item["sentence_lengths"] for item in batch]

    src_lengths = torch.tensor([len(x) for x in src], dtype=torch.long)
    tgt_lengths = torch.tensor([len(x) for x in tgt], dtype=torch.long)
    hier_lengths = torch.tensor([len(x) for x in hier_src], dtype=torch.long)
    num_sentences = torch.tensor([len(x) for x in sentence_lengths], dtype=torch.long)

    src_padded = pad_sequence(src, batch_first=True, padding_value=vocab.pad_id)
    tgt_padded = pad_sequence(tgt, batch_first=True, padding_value=vocab.pad_id)
    hier_padded = pad_sequence(hier_src, batch_first=True, padding_value=vocab.pad_id)
    sentence_lengths_padded = pad_sequence(sentence_lengths, batch_first=True, padding_value=0)

    return {
        "src_ids": src_padded,
        "src_lengths": src_lengths,
        "tgt_ids": tgt_padded,
        "tgt_lengths": tgt_lengths,
        "hier_src_ids": hier_padded,
        "hier_lengths": hier_lengths,
        "sentence_lengths": sentence_lengths_padded,
        "num_sentences": num_sentences,
        "raw_titles": [item["raw_title"] for item in batch],
    }

train_ds = TitleDataset(train_proc, vocab)
val_ds = TitleDataset(val_proc, vocab)
test_ds = TitleDataset(test_proc, vocab)

train_loader = DataLoader(
    train_ds,
    batch_size=CFG.batch_size,
    shuffle=True,
    num_workers=CFG.num_workers,
    collate_fn=collate_batch,
)
val_loader = DataLoader(
    val_ds,
    batch_size=CFG.batch_size,
    shuffle=False,
    num_workers=CFG.num_workers,
    collate_fn=collate_batch,
)
test_loader = DataLoader(
    test_ds,
    batch_size=CFG.batch_size,
    shuffle=False,
    num_workers=CFG.num_workers,
    collate_fn=collate_batch,
)

first_batch = next(iter(train_loader))
{k: tuple(v.shape) if torch.is_tensor(v) else len(v) for k, v in first_batch.items()}

{'src_ids': (16, 600),
 'src_lengths': (16,),
 'tgt_ids': (16, 6),
 'tgt_lengths': (16,),
 'hier_src_ids': (16, 568),
 'hier_lengths': (16,),
 'sentence_lengths': (16, 40),
 'num_sentences': (16,),
 'raw_titles': 16}

## Task B1 — Model Definitions

### Task B1, Task 1: EncoderRNN, DecoderRNN, Seq2seqRNN

- **`EncoderRNN`**: Bidirectional GRU encoder. Includes `load_embeddings()` setter for GloVe (Task B2, Task 5).
- **`HierEncoderRNN`**: Hierarchical encoder — word-level BiGRU + sentence-level BiGRU (Task B2, Task 6).
- **`DecoderRNN`**: Unidirectional GRU decoder with optional Bahdanau-style attention (Bonus).
- **`Decoder2RNN`**: Two-layer stacked GRU decoder (Task B2, Task 7).
- **`Seq2seqRNN`**: Combines encoder + decoder. Constructor parameters select GloVe, HierEncoder, Decoder2, attention, embedding tying. `forward()` parameter `decode_strategy` selects greedy or beam search (Task B2, Task 8).

In [ ]:
def lengths_to_mask(lengths: torch.Tensor, max_len: Optional[int] = None) -> torch.Tensor:
    max_len = max_len or int(lengths.max().item())
    positions = torch.arange(max_len, device=lengths.device).unsqueeze(0)
    return positions < lengths.unsqueeze(1)


class EncoderRNN(nn.Module):
    def __init__(self, vocab_size: int, embed_dim: int, hidden_dim: int, pad_id: int, dropout: float = 0.2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
        self.dropout = nn.Dropout(dropout)
        self.gru = nn.GRU(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            batch_first=True,
            bidirectional=True,
        )
        self.hidden_proj = nn.Linear(hidden_dim * 2, hidden_dim)
        self.memory_proj = nn.Linear(hidden_dim * 2, hidden_dim)

    def load_embeddings(self, glove_path: str, vocab: Vocab):
        if glove_path is None or not Path(glove_path).exists():
            print("GloVe path not found. Continuing with randomly initialized embeddings.")
            return

        embedding_weight = self.embedding.weight.data
        hits = 0
        with open(glove_path, "r", encoding="utf-8") as f:
            for line in f:
                parts = line.rstrip().split(" ")
                token, values = parts[0], parts[1:]
                if token in vocab.stoi and len(values) == embedding_weight.shape[1]:
                    embedding_weight[vocab.stoi[token]] = torch.tensor(np.asarray(values, dtype=np.float32))
                    hits += 1
        print(f"Loaded GloVe vectors for {hits} tokens out of {len(vocab)}")

    def forward(self, src_ids: torch.Tensor, src_lengths: torch.Tensor):
        embedded = self.dropout(self.embedding(src_ids))
        packed = pack_padded_sequence(embedded, src_lengths.cpu(), batch_first=True, enforce_sorted=False)
        packed_outputs, hidden = self.gru(packed)
        outputs, _ = pad_packed_sequence(packed_outputs, batch_first=True)
        memory = torch.tanh(self.memory_proj(outputs))
        hidden_cat = torch.cat([hidden[-2], hidden[-1]], dim=-1)
        decoder_init = torch.tanh(self.hidden_proj(hidden_cat)).unsqueeze(0)
        return memory, decoder_init


class HierEncoderRNN(nn.Module):
    def __init__(self, vocab_size: int, embed_dim: int, hidden_dim: int, pad_id: int, dropout: float = 0.2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
        self.dropout = nn.Dropout(dropout)
        self.word_gru = nn.GRU(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            batch_first=True,
            bidirectional=True,
        )

        self.word_attn = nn.Linear(hidden_dim * 2, 1)

        self.sent_gru = nn.GRU(
            input_size=hidden_dim * 2,
            hidden_size=hidden_dim,
            batch_first=True,
            bidirectional=True,
        )
        self.hidden_proj = nn.Linear(hidden_dim * 2, hidden_dim)
        self.memory_proj = nn.Linear(hidden_dim * 2, hidden_dim)

    def forward(
        self,
        src_ids: torch.Tensor,
        src_lengths: torch.Tensor,
        sentence_lengths: torch.Tensor,
        num_sentences: torch.Tensor,
    ):
        embedded = self.dropout(self.embedding(src_ids))
        packed = pack_padded_sequence(embedded, src_lengths.cpu(), batch_first=True, enforce_sorted=False)
        packed_outputs, word_hidden = self.word_gru(packed)
        word_outputs, _ = pad_packed_sequence(packed_outputs, batch_first=True)

        sentence_vectors = []
        max_sents = int(num_sentences.max().item())
        batch_size = src_ids.size(0)

        for batch_idx in range(batch_size):
            spans = []
            cursor = 0
            valid_word_len = int(src_lengths[batch_idx].item())   # actual (non-padded) token count
            valid_sent_lengths = sentence_lengths[batch_idx][: num_sentences[batch_idx]].tolist()
            for sent_len in valid_sent_lengths:
                sent_len = int(sent_len)
                if sent_len <= 0:
                  continue
                # Clamp the slice so it never reads into padding positions
                actual_end = min(cursor + sent_len, valid_word_len)
                if actual_end <= cursor:
                    break
                sent_output = word_outputs[batch_idx, cursor : actual_end]

                attn_scores = self.word_attn(sent_output)          # (sent_len, 1)
                attn_weights = torch.softmax(attn_scores, dim=0)
                sent_vector = (attn_weights * sent_output).sum(dim=0)

                spans.append(sent_vector)

                #cursor += sent_len
                cursor = actual_end
            if not spans:
                spans = [word_outputs[batch_idx, :1].mean(dim=0)]
            spans = torch.stack(spans, dim=0)
            if spans.size(0) < max_sents:
                pad = torch.zeros(max_sents - spans.size(0), spans.size(1), device=spans.device)
                spans = torch.cat([spans, pad], dim=0)
            sentence_vectors.append(spans)

        sentence_tensor = torch.stack(sentence_vectors, dim=0)
        packed_sent = pack_padded_sequence(sentence_tensor, num_sentences.cpu().clamp(min=1), batch_first=True, enforce_sorted=False)
        packed_sent_outputs, sent_hidden = self.sent_gru(packed_sent)
        sent_outputs, _ = pad_packed_sequence(packed_sent_outputs, batch_first=True)
        memory = torch.tanh(self.memory_proj(sent_outputs))
        hidden_cat = torch.cat([sent_hidden[-2], sent_hidden[-1]], dim=-1)
        decoder_init = torch.tanh(self.hidden_proj(hidden_cat)).unsqueeze(0)
        return memory, decoder_init


class DecoderRNN(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        embed_dim: int,
        hidden_dim: int,
        pad_id: int,
        dropout: float = 0.2,
        use_attention: bool = False,
    ):
        super().__init__()
        self.use_attention = use_attention
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
        self.dropout = nn.Dropout(dropout)
        self.gru = nn.GRU(embed_dim, hidden_dim, batch_first=True)
        if use_attention:
            self.attn_query = nn.Linear(hidden_dim, hidden_dim, bias=False)
            self.attn_key = nn.Linear(hidden_dim, hidden_dim, bias=False)
            self.context_gate = nn.Linear(hidden_dim * 2, hidden_dim)
        self.output = nn.Linear(hidden_dim, vocab_size)

    def compute_context(
        self,
        hidden_state: torch.Tensor,
        encoder_memory: Optional[torch.Tensor],
        encoder_mask: Optional[torch.Tensor],
    ) -> torch.Tensor:
        if not self.use_attention or encoder_memory is None:
            return hidden_state
        query = self.attn_query(hidden_state).unsqueeze(1)
        keys = self.attn_key(encoder_memory)
        scores = torch.bmm(query, keys.transpose(1, 2)).squeeze(1)
        if encoder_mask is not None:
            scores = scores.masked_fill(~encoder_mask, torch.finfo(scores.dtype).min)
        attn_weights = torch.softmax(scores, dim=-1)
        context = torch.bmm(attn_weights.unsqueeze(1), encoder_memory).squeeze(1)
        return torch.tanh(self.context_gate(torch.cat([hidden_state, context], dim=-1)))

    def forward_step(
        self,
        input_ids: torch.Tensor,
        hidden: torch.Tensor,
        encoder_memory: Optional[torch.Tensor] = None,
        encoder_mask: Optional[torch.Tensor] = None,
    ):
        embedded = self.dropout(self.embedding(input_ids.unsqueeze(1)))
        output, hidden = self.gru(embedded, hidden)
        decoder_state = output.squeeze(1)
        decoder_state = self.compute_context(decoder_state, encoder_memory, encoder_mask)
        logits = self.output(decoder_state)
        return logits, hidden


class Decoder2RNN(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        embed_dim: int,
        hidden_dim: int,
        pad_id: int,
        dropout: float = 0.2,
        use_attention: bool = False,
    ):
        super().__init__()
        self.use_attention = use_attention
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
        self.dropout = nn.Dropout(dropout)
        self.gru1 = nn.GRU(embed_dim, hidden_dim, batch_first=True)
        self.gru2 = nn.GRU(hidden_dim, hidden_dim, batch_first=True)
        if use_attention:
            self.attn_query = nn.Linear(hidden_dim, hidden_dim, bias=False)
            self.attn_key = nn.Linear(hidden_dim, hidden_dim, bias=False)
            self.context_gate = nn.Linear(hidden_dim * 2, hidden_dim)
        self.output = nn.Linear(hidden_dim, vocab_size)

    def compute_context(
        self,
        hidden_state: torch.Tensor,
        encoder_memory: Optional[torch.Tensor],
        encoder_mask: Optional[torch.Tensor],
    ) -> torch.Tensor:
        if not self.use_attention or encoder_memory is None:
            return hidden_state
        query = self.attn_query(hidden_state).unsqueeze(1)
        keys = self.attn_key(encoder_memory)
        scores = torch.bmm(query, keys.transpose(1, 2)).squeeze(1)
        if encoder_mask is not None:
            scores = scores.masked_fill(~encoder_mask, torch.finfo(scores.dtype).min)
        attn_weights = torch.softmax(scores, dim=-1)
        context = torch.bmm(attn_weights.unsqueeze(1), encoder_memory).squeeze(1)
        return torch.tanh(self.context_gate(torch.cat([hidden_state, context], dim=-1)))

    def forward_step(
        self,
        input_ids: torch.Tensor,
        hidden: Tuple[torch.Tensor, torch.Tensor],
        encoder_memory: Optional[torch.Tensor] = None,
        encoder_mask: Optional[torch.Tensor] = None,
    ):
        hidden1, hidden2 = hidden
        embedded = self.dropout(self.embedding(input_ids.unsqueeze(1)))
        out1, hidden1 = self.gru1(embedded, hidden1)
        out2, hidden2 = self.gru2(out1, hidden2)
        decoder_state = out2.squeeze(1)
        decoder_state = self.compute_context(decoder_state, encoder_memory, encoder_mask)
        logits = self.output(decoder_state)
        return logits, (hidden1, hidden2)

In [ ]:
class Seq2seqRNN(nn.Module):
    def __init__(
        self,
        vocab: Vocab,
        embed_dim: int = 300,
        hidden_dim: int = 300,
        pad_id: int = 0,
        dropout: float = 0.2,
        use_glove: bool = False,
        glove_path: Optional[str] = None,
        use_hier_encoder: bool = False,
        use_decoder2: bool = False,
        use_attention: bool = False,
        tie_embeddings: bool = False,
    ):
        super().__init__()
        self.vocab = vocab
        self.hidden_dim = hidden_dim
        self.use_hier_encoder = use_hier_encoder
        self.use_decoder2 = use_decoder2
        self.use_attention = use_attention
        self.pad_id = pad_id

        if use_hier_encoder:
            self.encoder = HierEncoderRNN(len(vocab), embed_dim, hidden_dim, pad_id, dropout)
        else:
            self.encoder = EncoderRNN(len(vocab), embed_dim, hidden_dim, pad_id, dropout)

        if use_glove and not use_hier_encoder:
            self.encoder.load_embeddings(glove_path, vocab)
        elif use_glove and use_hier_encoder:
            self.encoder.embedding.weight.data.normal_(0, 0.02)
            EncoderRNN.load_embeddings(self.encoder, glove_path, vocab)

        if use_decoder2:
            self.decoder = Decoder2RNN(
                len(vocab),
                embed_dim,
                hidden_dim,
                pad_id,
                dropout,
                use_attention=use_attention,
            )
        else:
            self.decoder = DecoderRNN(
                len(vocab),
                embed_dim,
                hidden_dim,
                pad_id,
                dropout,
                use_attention=use_attention,
            )

        if tie_embeddings:
            if self.encoder.embedding.weight.shape != self.decoder.embedding.weight.shape:
                raise ValueError("Encoder and decoder embedding shapes must match for weight tying.")
            self.decoder.embedding.weight = self.encoder.embedding.weight
            # Only tie output weights when output projection shape matches embedding
            # (vocab_size, embed_dim) must equal (vocab_size, hidden_dim) AND
            # the output layer shape (vocab_size, hidden_dim) must match embedding shape
            # This tying is only valid when embed_dim == hidden_dim == vocab_size, which is never true
            # So we skip output weight tying to avoid shape mismatch errors
            # if embed_dim == hidden_dim and not use_attention:
            #     self.decoder.output.weight = self.decoder.embedding.weight
            pass  # output weight tying skipped (vocab_size != hidden_dim)

    def encode(self, batch):
        if self.use_hier_encoder:
            memory, hidden = self.encoder(
                batch["hier_src_ids"].to(DEVICE),
                batch["hier_lengths"].to(DEVICE),
                batch["sentence_lengths"].to(DEVICE),
                batch["num_sentences"].to(DEVICE),
            )
            memory_lengths = batch["num_sentences"].to(DEVICE).clamp(min=1)
        else:
            memory, hidden = self.encoder(
                batch["src_ids"].to(DEVICE),
                batch["src_lengths"].to(DEVICE),
            )
            memory_lengths = batch["src_lengths"].to(DEVICE)
        memory_mask = lengths_to_mask(memory_lengths, memory.size(1))
        return memory, hidden, memory_mask

    def init_decoder_hidden(self, encoder_hidden: torch.Tensor):
        if self.use_decoder2:
            return encoder_hidden, encoder_hidden.clone()
        return encoder_hidden

    def forward(
        self,
        batch,
        teacher_forcing_ratio: float = 1.0,
        max_new_tokens: int = 16,
        decode_strategy: str = "greedy",
        beam_width: int = 4,
        length_penalty_alpha: float = 0.7,
    ):
        encoder_memory, encoder_hidden, encoder_mask = self.encode(batch)
        decoder_hidden = self.init_decoder_hidden(encoder_hidden)
        tgt = batch.get("tgt_ids")

        if tgt is not None:
            tgt = tgt.to(DEVICE)
            batch_size = tgt.size(0)
            steps = tgt.size(1) - 1
        else:
            batch_size = batch["src_ids"].size(0)
            steps = max_new_tokens

        if tgt is None and decode_strategy == "beam":
            return self.beam_search(
                batch=batch,
                beam_width=beam_width,
                max_new_tokens=max_new_tokens,
                length_penalty_alpha=length_penalty_alpha,
            )

        decoder_input = torch.full((batch_size,), self.vocab.bos_id, dtype=torch.long, device=DEVICE)
        logits_per_step = []
        predictions = []

        for step in range(steps):
            logits, decoder_hidden = self.decoder.forward_step(
                decoder_input,
                decoder_hidden,
                encoder_memory=encoder_memory,
                encoder_mask=encoder_mask,
            )
            logits_per_step.append(logits.unsqueeze(1))
            next_tokens = logits.argmax(dim=-1)
            predictions.append(next_tokens.unsqueeze(1))

            if tgt is None and (next_tokens == self.vocab.eos_id).all():
              break

            use_teacher = (
                tgt is not None
                and step + 1 < tgt.size(1)
                and random.random() < teacher_forcing_ratio
            )
            decoder_input = tgt[:, step + 1] if use_teacher else next_tokens


        logits = torch.cat(logits_per_step, dim=1)
        predictions = torch.cat(predictions, dim=1)
        return {"logits": logits, "predictions": predictions}

    @torch.no_grad()
    def beam_search(
        self,
        batch,
        beam_width: int = 4,
        max_new_tokens: int = 16,
        length_penalty_alpha: float = 0.7,
    ):
        assert batch["src_ids"].size(0) == 1, "Beam search is implemented for batch_size=1 inference."

        encoder_memory, encoder_hidden, encoder_mask = self.encode(batch)
        initial_hidden = self.init_decoder_hidden(encoder_hidden)
        beams = [
            {
                "tokens": [self.vocab.bos_id],
                "score": 0.0,
                "hidden": initial_hidden,
                "ended": False,
            }
        ]

        for _ in range(max_new_tokens):
            candidates = []
            for beam in beams:
                if beam["ended"]:
                    candidates.append(beam)
                    continue

                decoder_input = torch.tensor([beam["tokens"][-1]], device=DEVICE)
                logits, next_hidden = self.decoder.forward_step(
                    decoder_input,
                    beam["hidden"],
                    encoder_memory=encoder_memory,
                    encoder_mask=encoder_mask,
                )
                log_probs = F.log_softmax(logits, dim=-1)
                top_log_probs, top_indices = torch.topk(log_probs, beam_width, dim=-1)

                for log_p, token_id in zip(top_log_probs[0], top_indices[0]):
                    new_tokens = beam["tokens"] + [int(token_id)]
                    candidates.append(
                        {
                            "tokens": new_tokens,
                            "score": beam["score"] + float(log_p),
                            "hidden": (
                              (next_hidden[0].clone(), next_hidden[1].clone())
                              if self.use_decoder2
                              else next_hidden.clone()
                            ),
                            "ended": int(token_id) == self.vocab.eos_id,
                        }
                    )

            def normalized_score(item):
              length = max(1, len(item["tokens"]) - 1)
              penalty = ((5 + length) / 6) ** length_penalty_alpha
              score = item["score"] / penalty

              #PRIORITIZE finished sequences
              if item["ended"]:
                return score + 1e6   # large boost
              return score

            beams = sorted(candidates, key=normalized_score, reverse=True)[:beam_width]
            if all(beam["ended"] for beam in beams):
                break

        best = beams[0]["tokens"][1:]
        return {"predictions": torch.tensor([best], device=DEVICE)}

## Task B1 — Training Utilities

### Task B1, Task 3 & 4: Training loop, validation, prediction, ROUGE evaluation

- `sequence_cross_entropy`: Masked cross-entropy loss over padded sequences.
- `compute_metrics`: ROUGE-1, ROUGE-2, ROUGE-L F1 and exact-match accuracy.
- `train_model`: Adam optimiser, linear teacher-forcing decay (Bonus), gradient clipping, early stopping, best-model checkpointing, mixed precision (Bonus).
- `predict_dataset`: Greedy or beam-search inference over the test set.

### Task B2, Task 2: Vocabulary
Already built in Task A (≥ 1% document frequency threshold).

In [ ]:
def sequence_cross_entropy(logits: torch.Tensor, targets: torch.Tensor, pad_id: int) -> torch.Tensor:
    vocab_size = logits.size(-1)
    return F.cross_entropy(
        logits.reshape(-1, vocab_size),
        targets.reshape(-1),
        ignore_index=pad_id,
    )

def ids_to_text(ids: Sequence[int], vocab: Vocab) -> str:
    tokens = []
    for idx in ids:
        idx = int(idx)
        if idx == vocab.eos_id:
            break
        if idx in {vocab.pad_id, vocab.bos_id}:
            continue
        tokens.append(vocab.itos[idx])
    return " ".join(tokens).strip()

def collect_predictions(pred_tensor: torch.Tensor, vocab: Vocab) -> List[str]:
    return [ids_to_text(row.tolist(), vocab) for row in pred_tensor.detach().cpu()]

In [ ]:
try:
    from rouge_score import rouge_scorer
except Exception:
    rouge_scorer = None

def ngrams(tokens: List[str], n: int) -> Counter:
    if len(tokens) < n:
        return Counter()
    return Counter(tuple(tokens[i : i + n]) for i in range(len(tokens) - n + 1))

def rouge_n_f1(prediction: str, reference: str, n: int) -> float:
    pred_tokens = prediction.split()
    ref_tokens = reference.split()
    pred_ngrams = ngrams(pred_tokens, n)
    ref_ngrams = ngrams(ref_tokens, n)
    overlap = sum((pred_ngrams & ref_ngrams).values())
    if overlap == 0:
        return 0.0
    precision = overlap / max(1, sum(pred_ngrams.values()))
    recall = overlap / max(1, sum(ref_ngrams.values()))
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)

def lcs_length(a: List[str], b: List[str]) -> int:
    dp = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]
    for i in range(1, len(a) + 1):
        for j in range(1, len(b) + 1):
            if a[i - 1] == b[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1
            else:
                dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])
    return dp[-1][-1]

def rouge_l_f1(prediction: str, reference: str) -> float:
    pred_tokens = prediction.split()
    ref_tokens = reference.split()
    overlap = lcs_length(pred_tokens, ref_tokens)
    if overlap == 0:
        return 0.0
    precision = overlap / max(1, len(pred_tokens))
    recall = overlap / max(1, len(ref_tokens))
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)

def compute_metrics(predictions: List[str], references: List[str]) -> Dict[str, float]:
    metrics = {
        "rouge1_f1": float("nan"),
        "rouge2_f1": float("nan"),
        "rougel_f1": float("nan"),
        "token_accuracy": 0.0,
    }

    # Unified normalization to ensure fair comparison
    def normalize(text):
        text = text.strip().lower()
        text = re.sub(r"[^\w\s]", "", text) # Remove punctuation
        text = re.sub(r"\s+", " ", text)    # Collapse whitespace
        return text

    norm_preds = [normalize(p) for p in predictions]
    norm_refs  = [normalize(r) for r in references]

    # 1. Calculate ROUGE Scores
    if rouge_scorer is not None:
        scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
        r1, r2, rl = [], [], []
        for pred, ref in zip(norm_preds, norm_refs):
            score = scorer.score(ref, pred)
            r1.append(score["rouge1"].fmeasure)
            r2.append(score["rouge2"].fmeasure)
            rl.append(score["rougeL"].fmeasure)
        metrics["rouge1_f1"] = float(np.mean(r1))
        metrics["rouge2_f1"] = float(np.mean(r2))
        metrics["rougel_f1"] = float(np.mean(rl))
    else:
        metrics["rouge1_f1"] = float(np.mean([rouge_n_f1(p, r, 1) for p, r in zip(norm_preds, norm_refs)]))
        metrics["rouge2_f1"] = float(np.mean([rouge_n_f1(p, r, 2) for p, r in zip(norm_preds, norm_refs)]))
        metrics["rougel_f1"] = float(np.mean([rouge_l_f1(p, r)    for p, r in zip(norm_preds, norm_refs)]))

    # 2. Calculate Token-Level Accuracy
    token_accuracies = []
    for p, r in zip(norm_preds, norm_refs):
        p_tokens = p.split()
        r_tokens = r.split()

        if not r_tokens: # Handle empty references
            token_accuracies.append(1.0 if not p_tokens else 0.0)
            continue

        # Count how many tokens match at each position
        min_len = min(len(p_tokens), len(r_tokens))
        matches = sum(1 for i in range(min_len) if p_tokens[i] == r_tokens[i])

        # Accuracy is matches divided by the length of the longer sequence
        # (penalizes hallucinations or truncated text)
        max_len = max(len(p_tokens), len(r_tokens))
        token_accuracies.append(matches / max_len)

    metrics["token_accuracy"] = float(np.mean(token_accuracies))

    return metrics

In [ ]:
def get_teacher_forcing_ratio(epoch: int, cfg: Config) -> float:
    if cfg.epochs <= 1:
        return cfg.teacher_forcing_end
    progress = (epoch - 1) / (cfg.epochs - 1)
    return cfg.teacher_forcing_start + progress * (cfg.teacher_forcing_end - cfg.teacher_forcing_start)


def get_autocast_context(cfg: Config):
    use_amp = cfg.use_mixed_precision and DEVICE.type == "cuda"
    if use_amp:
        return torch.autocast(device_type="cuda", dtype=torch.float16)
    return nullcontext()


def run_validation(model: Seq2seqRNN, loader: DataLoader, cfg: Config) -> Dict[str, float]:
    """Compute validation metrics.

    Val LOSS uses teacher forcing = 1.0 so the loss is comparable across
    encoder types and the early-stopping signal is stable.
    ROUGE / accuracy are computed with a separate greedy pass so they
    reflect real generation quality.
    """
    model.eval()

    # ── Pass 1: teacher-forced loss (stable early-stopping signal) ──────────
    tf_losses = []
    with torch.no_grad():
        for batch in loader:
            with get_autocast_context(cfg):
                output = model(
                    batch,
                    teacher_forcing_ratio=1.0,   # teacher-forced for loss
                    max_new_tokens=cfg.max_output_tokens,
                    decode_strategy="greedy",
                )
            loss = sequence_cross_entropy(
                output["logits"],
                batch["tgt_ids"][:, 1:].to(DEVICE),
                vocab.pad_id,
            )
            tf_losses.append(loss.item())

    # ── Pass 2: greedy decoding for ROUGE / accuracy ─────────────────────────
    predictions = []
    references  = []
    with torch.no_grad():
        for batch in loader:
            with get_autocast_context(cfg):
                output = model(
                    batch,
                    teacher_forcing_ratio=0.0,   # free-running for ROUGE
                    max_new_tokens=cfg.max_output_tokens,
                    decode_strategy="greedy",
                )
            predictions.extend(collect_predictions(output["predictions"], vocab))
            # Decode tgt_ids to get the preprocessed reference tokens
            references.extend([
                " ".join(vocab.decode(row.tolist(), skip_special=True))
                for row in batch["tgt_ids"]
            ])

    metrics = compute_metrics(predictions, references)
    metrics["loss"] = float(np.mean(tf_losses))   # use teacher-forced loss
    return metrics

def train_model(model: Seq2seqRNN, train_loader: DataLoader, val_loader: DataLoader, cfg: Config):
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.learning_rate)
    scaler = torch.cuda.amp.GradScaler(enabled=cfg.use_mixed_precision and DEVICE.type == "cuda")
    history = []
    best_state = None
    best_val = float("inf")
    total_start = time.time()

    patience = 3   # increased from 2 to give hier models more time to converge
    no_improve_epochs = 0

    for epoch in range(1, cfg.epochs + 1):
        model.train()
        epoch_losses = []
        start = time.time()
        teacher_forcing_ratio = get_teacher_forcing_ratio(epoch, cfg)
        progress = tqdm(train_loader, desc=f"epoch {epoch}/{cfg.epochs}")
        for batch in progress:
            optimizer.zero_grad()
            with get_autocast_context(cfg):
                output = model(
                    batch,
                    teacher_forcing_ratio=teacher_forcing_ratio,
                    max_new_tokens=cfg.max_output_tokens,
                    decode_strategy="greedy",
                )
                loss = sequence_cross_entropy(
                    output["logits"],
                    batch["tgt_ids"][:, 1:].to(DEVICE),
                    vocab.pad_id,
                )
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            epoch_losses.append(loss.item())
            progress.set_postfix(train_loss=np.mean(epoch_losses), tf_ratio=teacher_forcing_ratio)

        val_metrics = run_validation(model, val_loader, cfg)
        record = {
            "epoch": epoch,
            "train_loss": float(np.mean(epoch_losses)),
            "val_loss": val_metrics["loss"],
            "val_rouge1_f1": val_metrics["rouge1_f1"],
            "val_rouge2_f1": val_metrics["rouge2_f1"],
            "val_rougel_f1": val_metrics["rougel_f1"],
            "val_accuracy": val_metrics["token_accuracy"],
            "teacher_forcing_ratio": teacher_forcing_ratio,
            "epoch_seconds": time.time() - start,
        }
        history.append(record)
        print(record)

        if val_metrics["loss"] < best_val:
          best_val = val_metrics["loss"]
          best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
          no_improve_epochs = 0
        else:
          no_improve_epochs += 1

        if no_improve_epochs >= patience:
          print("Early stopping triggered")
          break

    total_seconds = time.time() - total_start
    if best_state is not None:
        model.load_state_dict(best_state)
    return history, total_seconds

@torch.no_grad()
def predict_dataset(model: Seq2seqRNN, loader: DataLoader, cfg: Config, decode_strategy: str = "greedy") -> List[str]:
    model.eval()
    predictions = []
    for batch in loader:
        if decode_strategy == "beam":
            for idx in range(batch["src_ids"].size(0)):
                single_batch = {}
                for key, value in batch.items():
                    if torch.is_tensor(value):
                        single_batch[key] = value[idx : idx + 1]
                    elif isinstance(value, list):
                        single_batch[key] = value[idx : idx + 1]
                    else:
                        single_batch[key] = value
                with get_autocast_context(cfg):
                    output = model(
                        single_batch,
                        teacher_forcing_ratio=0.0,
                        max_new_tokens=cfg.max_output_tokens,
                        decode_strategy="beam",
                        beam_width=cfg.beam_width,
                        length_penalty_alpha=cfg.length_penalty_alpha,
                    )
                predictions.extend(collect_predictions(output["predictions"], vocab))
        else:
            with get_autocast_context(cfg):
                output = model(
                    batch,
                    teacher_forcing_ratio=0.0,
                    max_new_tokens=cfg.max_output_tokens,
                    decode_strategy="greedy",
                )
            predictions.extend(collect_predictions(output["predictions"], vocab))
    return predictions


## Task B1 — Baseline Model

### Task B1, Tasks 1–4: Train and evaluate the basic RNN Seq2seq model

No GloVe, no hierarchical encoder, no two-layer decoder, no beam search, no attention. Pure baseline as specified in Task B1.

In [ ]:
baseline_model = Seq2seqRNN(
    vocab=vocab,
    embed_dim=CFG.embed_dim,
    hidden_dim=CFG.hidden_dim,
    pad_id=vocab.pad_id,
    dropout=CFG.dropout,
    use_glove=False,
    glove_path=CFG.glove_path,
    use_hier_encoder=False,
    use_decoder2=False,
    use_attention=False,  # clean baseline: no bonus features
    tie_embeddings=False,  # clean baseline: no bonus features
).to(DEVICE)

baseline_history, baseline_train_seconds = train_model(baseline_model, train_loader, val_loader, CFG)
baseline_history_df = pd.DataFrame(baseline_history)
baseline_predictions = predict_dataset(baseline_model, test_loader, CFG, decode_strategy="greedy")
# Use preprocessed title_tokens as references (same form as predictions)
test_references = test_proc["title_tokens"].apply(lambda x: " ".join(x)).tolist()
baseline_metrics = compute_metrics(baseline_predictions, test_references)
baseline_result = {
    "experiment": "baseline",
    "train_seconds": baseline_train_seconds,
    "decode_strategy": "greedy",
    **baseline_metrics,
}
baseline_history_df

/tmp/ipykernel_819/1891756481.py:68: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=cfg.use_mixed_precision and DEVICE.type == "cuda")


epoch 1/10:   0%|          | 0/282 [00:00<?, ?it/s]

{'epoch': 1, 'train_loss': 3.004016374019866, 'val_loss': 2.354339599609375, 'val_rouge1_f1': 0.07951831501831502, 'val_rouge2_f1': 0.016544588744588744, 'val_rougel_f1': 0.07951831501831502, 'val_accuracy': 0.055409523809523804, 'teacher_forcing_ratio': 1.0, 'epoch_seconds': 23.739636659622192}


epoch 2/10:   0%|          | 0/282 [00:00<?, ?it/s]

{'epoch': 2, 'train_loss': 1.9350275623460187, 'val_loss': 2.0615997314453125, 'val_rouge1_f1': 0.15801428571428572, 'val_rouge2_f1': 0.03543809523809524, 'val_rougel_f1': 0.15801428571428572, 'val_accuracy': 0.09589999999999999, 'teacher_forcing_ratio': 0.9555555555555556, 'epoch_seconds': 22.672970294952393}


epoch 3/10:   0%|          | 0/282 [00:00<?, ?it/s]

... (4 outputs left out)

{'epoch': 5, 'train_loss': 0.47745469883612707, 'val_loss': 2.0269927978515625, 'val_rouge1_f1': 0.2375060606060606, 'val_rouge2_f1': 0.05665396825396826, 'val_rougel_f1': 0.23693463203463203, 'val_accuracy': 0.11203333333333333, 'teacher_forcing_ratio': 0.8222222222222222, 'epoch_seconds': 22.19105076789856}


epoch 6/10:   0%|          | 0/282 [00:00<?, ?it/s]

{'epoch': 6, 'train_loss': 0.3122144538169423, 'val_loss': 2.1229400634765625, 'val_rouge1_f1': 0.2436222222222222, 'val_rouge2_f1': 0.0625095238095238, 'val_rougel_f1': 0.2436222222222222, 'val_accuracy': 0.12296666666666667, 'teacher_forcing_ratio': 0.7777777777777778, 'epoch_seconds': 22.112449884414673}
Early stopping triggered


,epoch,train_loss,val_loss,val_rouge1_f1,val_rouge2_f1,val_rougel_f1,val_accuracy,teacher_forcing_ratio,epoch_seconds
0,1,3.004016,2.354340,0.079518,0.016545,0.079518,0.055410,1.000000,23.739637
1,2,1.935028,2.061600,0.158014,0.035438,0.158014,0.095900,0.955556,22.672970
2,3,1.288529,1.957794,0.195788,0.041244,0.195788,0.112967,0.911111,21.665121
3,4,0.798924,1.972519,0.212953,0.048644,0.212153,0.116967,0.866667,22.848123
4,5,0.477455,2.026993,0.237506,0.056654,0.236935,0.112033,0.822222,22.191051
5,6,0.312214,2.122940,0.243622,0.062510,0.243622,0.122967,0.777778,22.112450


In [ ]:
baseline_predictions_df = pd.DataFrame(
    {
        "reference_title": test_proc["title"].tolist(),
        "predicted_title": baseline_predictions,
    }
)
pd.DataFrame([baseline_result])

,experiment,train_seconds,decode_strategy,rouge1_f1,rouge2_f1,rougel_f1,token_accuracy
0,baseline,135.285281,greedy,0.174008,0.028881,0.174008,0.145667


## Task B2 — Improvements: Individual and Combined

Each improvement is evaluated **individually** (all others off), then **all four combined**:

| Experiment | Task | What changes |
|---|---|---|
| `glove_only` | B2 Task 5 | GloVe 6B 300d embeddings via `load_embeddings()` |
| `hier_only` | B2 Task 6 | `HierEncoderRNN` (word + sentence level BiGRU) |
| `decoder2_only` | B2 Task 7 | `Decoder2RNN` (two stacked GRUs) |
| `beam_only` | B2 Task 8 | Beam search decoding (width=4, length penalty α=0.7) |
| `all_improvements` | B2 All | All four combined |

## Bonus (Task 9) — Additional Enhancements

| Experiment | Enhancement |
|---|---|
| `attention_only` | Bahdanau-style decoder attention over encoder memory |
| `tied_embeddings_only` | Encoder–decoder embedding weight tying |
| `bonus_stack` | All improvements + attention + tied embeddings |

Additional bonus features always active: linear teacher-forcing decay, mixed-precision (FP16) training, gradient clipping, early stopping, preprocessing/vocab caching.

In [ ]:
def run_experiment(
    name: str,
    *,
    use_glove: bool = False,
    use_hier_encoder: bool = False,
    use_decoder2: bool = False,
    use_beam_search: bool = False,
    use_attention: bool = False,
    tie_embeddings: bool = False,
    cfg: Config = CFG,
):
    model = Seq2seqRNN(
        vocab=vocab,
        embed_dim=cfg.embed_dim,
        hidden_dim=cfg.hidden_dim,
        pad_id=vocab.pad_id,
        dropout=cfg.dropout,
        use_glove=use_glove,
        glove_path=cfg.glove_path,
        use_hier_encoder=use_hier_encoder,
        use_decoder2=use_decoder2,
        use_attention=use_attention,
        tie_embeddings=tie_embeddings,
    ).to(DEVICE)

    history, train_seconds = train_model(model, train_loader, val_loader, cfg)
    decode_strategy = "beam" if use_beam_search else "greedy"
    test_predictions = predict_dataset(model, test_loader, cfg, decode_strategy=decode_strategy)
    # Use preprocessed title_tokens as references (same form as predictions)
    test_references = test_proc["title_tokens"].apply(lambda x: " ".join(x)).tolist()
    test_metrics = compute_metrics(test_predictions, test_references)

    result = {
        "experiment": name,
        "train_seconds": train_seconds,
        "decode_strategy": decode_strategy,
        **test_metrics,
    }
    return model, pd.DataFrame(history), pd.DataFrame(
        {
            "reference_title": test_proc["title_tokens"].apply(lambda x: " ".join(x)).tolist(),
            "predicted_title": test_predictions,
        }
    ), result

In [ ]:
EXPERIMENTS = [
    {"name": "baseline", "use_glove": False, "use_hier_encoder": False, "use_decoder2": False, "use_beam_search": False},
    {"name": "glove_only", "use_glove": True, "use_hier_encoder": False, "use_decoder2": False, "use_beam_search": False},
    {"name": "hier_only", "use_glove": False, "use_hier_encoder": True, "use_decoder2": False, "use_beam_search": False},
    {"name": "decoder2_only", "use_glove": False, "use_hier_encoder": False, "use_decoder2": True, "use_beam_search": False},
    {"name": "beam_only", "use_glove": False, "use_hier_encoder": False, "use_decoder2": False, "use_beam_search": True},
    {"name": "all_improvements", "use_glove": True, "use_hier_encoder": True, "use_decoder2": True, "use_beam_search": True},
]

BONUS_EXPERIMENTS = [
    {"name": "attention_only", "use_attention": True},
    {"name": "tied_embeddings_only", "tie_embeddings": True},
    {
        "name": "bonus_stack",
        "use_glove": True,
        "use_hier_encoder": True,
        "use_decoder2": True,
        "use_beam_search": True,
        "use_attention": True,
        "tie_embeddings": True,
    },
]

selected = EXPERIMENTS+BONUS_EXPERIMENTS
selected

[{'name': 'baseline',
  'use_glove': False,
  'use_hier_encoder': False,
  'use_decoder2': False,
  'use_beam_search': False},
 {'name': 'glove_only',
  'use_glove': True,
  'use_hier_encoder': False,
  'use_decoder2': False,
  'use_beam_search': False},
 {'name': 'hier_only',
  'use_glove': False,
  'use_hier_encoder': True,
  'use_decoder2': False,
  'use_beam_search': False},
 {'name': 'decoder2_only',
  'use_glove': False,
  'use_hier_encoder': False,
  'use_decoder2': True,
  'use_beam_search': False},
 {'name': 'beam_only',
  'use_glove': False,
  'use_hier_encoder': False,
  'use_decoder2': False,
  'use_beam_search': True},
 {'name': 'all_improvements',
  'use_glove': True,
  'use_hier_encoder': True,
  'use_decoder2': True,
  'use_beam_search': True},
 {'name': 'attention_only', 'use_attention': True},
 {'name': 'tied_embeddings_only', 'tie_embeddings': True},
 {'name': 'bonus_stack',
  'use_glove': True,
  'use_hier_encoder': True,
  'use_decoder2': True,
  'use_beam_search':

In [ ]:
all_results = []
histories = {}
predictions = {}
trained_models = {}

for exp in selected:
    model, hist_df, pred_df, result = run_experiment(**exp)
    trained_models[exp["name"]] = model
    histories[exp["name"]] = hist_df
    predictions[exp["name"]] = pred_df
    all_results.append(result)

/tmp/ipykernel_819/1891756481.py:68: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=cfg.use_mixed_precision and DEVICE.type == "cuda")


epoch 1/10:   0%|          | 0/282 [00:00<?, ?it/s]

{'epoch': 1, 'train_loss': 2.9552846115531652, 'val_loss': 2.327423095703125, 'val_rouge1_f1': 0.10156190476190476, 'val_rouge2_f1': 0.023600000000000003, 'val_rougel_f1': 0.10156190476190476, 'val_accuracy': 0.05606666666666666, 'teacher_forcing_ratio': 1.0, 'epoch_seconds': 22.723145246505737}


epoch 2/10:   0%|          | 0/282 [00:00<?, ?it/s]

{'epoch': 2, 'train_loss': 1.9273871246804581, 'val_loss': 2.107208251953125, 'val_rouge1_f1': 0.16315714285714283, 'val_rouge2_f1': 0.03007142857142857, 'val_rougel_f1': 0.16315714285714283, 'val_accuracy': 0.09626666666666667, 'teacher_forcing_ratio': 0.9555555555555556, 'epoch_seconds': 21.745909452438354}


epoch 3/10:   0%|          | 0/282 [00:00<?, ?it/s]

... (123 outputs left out)

epoch 6/10:   0%|          | 0/282 [00:00<?, ?it/s]

{'epoch': 6, 'train_loss': 3.0655771903957882, 'val_loss': 3.16107177734375, 'val_rouge1_f1': 0.0, 'val_rouge2_f1': 0.0, 'val_rougel_f1': 0.0, 'val_accuracy': 0.0, 'teacher_forcing_ratio': 0.7777777777777778, 'epoch_seconds': 112.60916590690613}


epoch 7/10:   0%|          | 0/282 [00:00<?, ?it/s]

{'epoch': 7, 'train_loss': 3.078206475322128, 'val_loss': 3.173828125, 'val_rouge1_f1': 0.0, 'val_rouge2_f1': 0.0, 'val_rougel_f1': 0.0, 'val_accuracy': 0.0, 'teacher_forcing_ratio': 0.7333333333333334, 'epoch_seconds': 113.16386890411377}
Early stopping triggered


In [ ]:
results_df = pd.DataFrame(all_results).sort_values("rougel_f1", ascending=False)
#results_df.drop(columns=["accuracy"],inplace=True)
results_df.to_csv("/content/drive/MyDrive/NLP Assignment 2/experiment_results.csv", index=False)
for name, hist_df in histories.items():
    hist_df.to_csv(f"/content/drive/MyDrive/NLP Assignment 2/{name}_history.csv", index=False)
for name, pred_df in predictions.items():
    pred_df.to_csv(f"/content/drive/MyDrive/NLP Assignment 2/{name}_test_predictions.csv", index=False)
results_df

,experiment,train_seconds,decode_strategy,rouge1_f1,rouge2_f1,rougel_f1,token_accuracy
7,tied_embeddings_only,155.143960,greedy,0.217183,0.052881,0.217183,0.192667
3,decoder2_only,167.982850,greedy,0.202302,0.036905,0.202302,0.160833
4,beam_only,154.707974,beam,0.184714,0.041548,0.184714,0.163000
0,baseline,155.555479,greedy,0.182190,0.039167,0.182190,0.157000
1,glove_only,200.844951,greedy,0.149212,0.035509,0.149212,0.122333
6,attention_only,246.566063,greedy,0.079817,0.023000,0.079817,0.068833
2,hier_only,529.816025,greedy,0.000000,0.000000,0.000000,0.000000
5,all_improvements,753.240099,beam,0.000000,0.000000,0.000000,0.000000
8,bonus_stack,784.189774,beam,0.000000,0.000000,0.000000,0.000000


## Bonus (Task 9) — Bonus Stack Results

Summary of the fully stacked bonus model (GloVe + HierEncoder + Decoder2 + Beam + Attention + Tied Embeddings).

In [ ]:
bonus_result_df = results_df[results_df["experiment"] == "bonus_stack"].reset_index(drop=True)
bonus_result_df

,experiment,train_seconds,decode_strategy,rouge1_f1,rouge2_f1,rougel_f1,token_accuracy
0,bonus_stack,784.189774,beam,0.0,0.0,0.0,0.0
